In [1]:
// Reference the necessary packages
#r "nuget: Aevatar.Core.Abstractions, *-*, Source=https://www.myget.org/F/aelf-project-dev/api/v3/index.json"
#r "nuget: Aevatar.Core, *-*, Source=https://www.myget.org/F/aelf-project-dev/api/v3/index.json"

using Aevatar.Core;
using Aevatar.Core.Abstractions;
using Orleans;
using Microsoft.Extensions.Logging;
using System;
using System.Threading.Tasks;
using System.Collections.Generic;

Console.WriteLine("✅ Packages loaded successfully!");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Aevatar.Core, 1.5.2 Aevatar.Core.Abstractions, 1.5.2

✅ Packages loaded successfully!


In [2]:
/// <summary>
/// State class for our simple greeting GAgent
/// </summary>
[GenerateSerializer]
public class GreetingGAgentState : StateBase
{
    /// <summary>
    /// Store the greeting message context
    /// </summary>
    [Id(0)] public string Context { get; set; } = string.Empty;
    
    /// <summary>
    /// Track when the last greeting was received
    /// </summary>
    [Id(1)] public DateTime LastGreetingTime { get; set; }
    
    /// <summary>
    /// Keep a history of all greetings received
    /// </summary>
    [Id(2)] public List<string> GreetingHistory { get; set; } = new();
}

Console.WriteLine("✅ State class defined!");


✅ State class defined!


In [4]:
/// <summary>
/// Base class for all state log events in our GAgent
/// </summary>
[GenerateSerializer]
public class GreetingStateLogEvent : StateLogEventBase<GreetingStateLogEvent>;

/// <summary>
/// Event that stores a greeting in the state
/// </summary>
[GenerateSerializer]
public class StoreGreetingLogEvent : GreetingStateLogEvent
{
    [Id(0)] public string Greeting { get; init; } = string.Empty;
    [Id(1)] public DateTime Timestamp { get; init; } = DateTime.UtcNow;
}

Console.WriteLine("✅ State log events defined!");


✅ State log events defined!


In [6]:
/// <summary>
/// External event that contains a greeting message
/// </summary>
[GenerateSerializer]
public class GreetingEvent : EventBase
{
    /// <summary>
    /// The greeting message to store
    /// </summary>
    [Id(0)] 
    [System.ComponentModel.Description("A friendly greeting message to be stored in the GAgent's context")]
    public string Message { get; init; } = string.Empty;
    
    /// <summary>
    /// Optional sender name
    /// </summary>
    [Id(1)]
    [System.ComponentModel.Description("Name of the person or agent sending the greeting")]
    public string Sender { get; init; } = "Anonymous";
}

Console.WriteLine("✅ External events defined!");


✅ External events defined!


In [7]:
/// <summary>
/// Interface for our simple greeting GAgent
/// </summary>
public interface IGreetingGAgent : IStateGAgent<GreetingGAgentState>
{
    /// <summary>
    /// Store a greeting message in the context
    /// </summary>
    /// <param name="greeting">The greeting message to store</param>
    /// <param name="sender">Who sent the greeting</param>
    /// <returns>True if stored successfully</returns>
    Task<bool> StoreGreetingAsync(string greeting, string sender = "Anonymous");
    
    /// <summary>
    /// Get the current greeting context
    /// </summary>
    /// <returns>The current greeting stored in context</returns>
    Task<string> GetCurrentGreetingAsync();
    
    /// <summary>
    /// Get the history of all greetings
    /// </summary>
    /// <returns>List of all greetings received</returns>
    Task<List<string>> GetGreetingHistoryAsync();
}

Console.WriteLine("✅ Interface defined!");


✅ Interface defined!


In [8]:
/// <summary>
/// Simple GAgent that demonstrates greeting storage with event handling
/// </summary>
[GAgent("greeting", "demo")]
public class GreetingGAgent : GAgentBase<GreetingGAgentState, GreetingStateLogEvent>, IGreetingGAgent
{
    /// <summary>
    /// Required method - describes what this GAgent does
    /// </summary>
    public override Task<string> GetDescriptionAsync()
    {
        return Task.FromResult(
            "A simple demonstration GAgent that stores greeting messages in its context. " +
            "It can receive greetings through events and store them in its state for later retrieval.");
    }

    // ========== PUBLIC INTERFACE METHODS ==========
    
    /// <summary>
    /// Store a greeting message in the context
    /// </summary>
    public async Task<bool> StoreGreetingAsync(string greeting, string sender = "Anonymous")
    {
        if (string.IsNullOrWhiteSpace(greeting))
            return false;

        try
        {
            // Use RaiseEvent to modify state - NEVER modify State directly!
            RaiseEvent(new StoreGreetingLogEvent 
            { 
                Greeting = $"[{sender}]: {greeting}",
                Timestamp = DateTime.UtcNow
            });
            
            // Commit the event to state
            await ConfirmEvents();
            
            Logger.LogInformation("Greeting stored successfully: {Greeting}", greeting);
            return true;
        }
        catch (Exception ex)
        {
            Logger.LogError(ex, "Failed to store greeting: {Greeting}", greeting);
            return false;
        }
    }

    /// <summary>
    /// Get the current greeting context
    /// </summary>
    public Task<string> GetCurrentGreetingAsync()
    {
        return Task.FromResult(State.Context);
    }

    /// <summary>
    /// Get the history of all greetings
    /// </summary>
    public Task<List<string>> GetGreetingHistoryAsync()
    {
        return Task.FromResult(new List<string>(State.GreetingHistory));
    }

    // ========== EVENT HANDLERS ==========
    
    /// <summary>
    /// Event handler that processes incoming greeting events
    /// This is the core functionality - storing string greetings in State.Context!
    /// </summary>
    [EventHandler]
    public async Task HandleGreetingEventAsync(GreetingEvent greetingEvent)
    {
        Logger.LogInformation("Received greeting event from {Sender}: {Message}", 
            greetingEvent.Sender, greetingEvent.Message);

        // Store the greeting using our public method
        await StoreGreetingAsync(greetingEvent.Message, greetingEvent.Sender);
        
        Logger.LogInformation("Greeting event processed successfully");
    }

    // ========== STATE MANAGEMENT ==========
    
    /// <summary>
    /// Handle state transitions when events are applied
    /// This is where the actual state changes happen!
    /// </summary>
    protected override void GAgentTransitionState(GreetingGAgentState state, StateLogEventBase<GreetingStateLogEvent> @event)
    {
        switch (@event)
        {
            case StoreGreetingLogEvent storeEvent:
                // Store in context (this is what the tutorial asked for!)
                state.Context = storeEvent.Greeting;
                state.LastGreetingTime = storeEvent.Timestamp;
                
                // Also add to history
                state.GreetingHistory.Add(storeEvent.Greeting);
                
                Logger.LogDebug("State updated - Context: {Context}, HistoryCount: {Count}", 
                    state.Context, state.GreetingHistory.Count);
                break;
            
            default:
                Logger.LogWarning("Unknown state log event type: {EventType}", @event.GetType().Name);
                break;
        }
    }
}

Console.WriteLine("✅ GAgent implementation complete!");


✅ GAgent implementation complete!


In [10]:
/// <summary>
/// Simple test class to verify our GAgent works
/// </summary>
public static class GreetingGAgentTest
{
    public static async Task RunBasicTests()
    {
        Console.WriteLine("🧪 Starting GAgent Tests...");
        Console.WriteLine();
        
        // Note: In a real scenario, you'd use Orleans TestKit or similar
        // For this demo, we'll test the state transition logic directly
        
        // Test 1: State Transition Logic
        Console.WriteLine("Test 1: State Transition Logic");
        var state = new GreetingGAgentState();
        var storeEvent = new StoreGreetingLogEvent 
        { 
            Greeting = "[Alice]: Hello, World!",
            Timestamp = DateTime.UtcNow
        };
        
        // Simulate the state transition (normally handled by Orleans)
        Console.WriteLine($"  Initial state - Context: '{state.Context}'");
        Console.WriteLine($"  Initial state - History count: {state.GreetingHistory.Count}");
        
        // Apply the event (this simulates what GAgentTransitionState does)
        state.Context = storeEvent.Greeting;
        state.LastGreetingTime = storeEvent.Timestamp;
        state.GreetingHistory.Add(storeEvent.Greeting);
        
        Console.WriteLine($"  After event - Context: '{state.Context}'");
        Console.WriteLine($"  After event - History count: {state.GreetingHistory.Count}");
        Console.WriteLine($"  ✅ State transition test passed!");
        Console.WriteLine();
        
        // Test 2: Event Structure
        Console.WriteLine("Test 2: Event Structure Validation");
        var greetingEvent = new GreetingEvent 
        { 
            Message = "Hello from the notebook!",
            Sender = "Tutorial"
        };
        
        Console.WriteLine($"  Event Message: '{greetingEvent.Message}'");
        Console.WriteLine($"  Event Sender: '{greetingEvent.Sender}'");
        Console.WriteLine($"  ✅ Event structure test passed!");
        Console.WriteLine();
        
        // Test 3: Multiple Events
        Console.WriteLine("Test 3: Multiple Events Handling");
        var greetings = new[] 
        {
            ("Alice", "Hello!"),
            ("Bob", "Hi there!"),
            ("Charlie", "Good morning!")
        };
        
        var testState = new GreetingGAgentState();
        
        foreach (var (sender, message) in greetings)
        {
            var evt = new StoreGreetingLogEvent
            {
                Greeting = $"[{sender}]: {message}",
                Timestamp = DateTime.UtcNow
            };
            
            // Apply state transition
            testState.Context = evt.Greeting;
            testState.LastGreetingTime = evt.Timestamp;
            testState.GreetingHistory.Add(evt.Greeting);
            
            Console.WriteLine($"  Processed: {evt.Greeting}");
        }
        
        Console.WriteLine($"  Final context: '{testState.Context}'");
        Console.WriteLine($"  Total greetings in history: {testState.GreetingHistory.Count}");
        Console.WriteLine($"  ✅ Multiple events test passed!");
        Console.WriteLine();
        
        Console.WriteLine("🎉 All tests completed successfully!");
    }
}

// Run the tests
await GreetingGAgentTest.RunBasicTests();


🧪 Starting GAgent Tests...

Test 1: State Transition Logic
  Initial state - Context: ''
  Initial state - History count: 0
  After event - Context: '[Alice]: Hello, World!'
  After event - History count: 1
  ✅ State transition test passed!

Test 2: Event Structure Validation
  Event Message: 'Hello from the notebook!'
  Event Sender: 'Tutorial'
  ✅ Event structure test passed!

Test 3: Multiple Events Handling
  Processed: [Alice]: Hello!
  Processed: [Bob]: Hi there!
  Processed: [Charlie]: Good morning!
  Final context: '[Charlie]: Good morning!'
  Total greetings in history: 3
  ✅ Multiple events test passed!

🎉 All tests completed successfully!



(6,30): warning CS1998: 此异步方法缺少 "await" 运算符，将以同步方式运行。请考虑使用 "await" 运算符等待非阻止的 API 调用，或者使用 "await Task.Run(...)" 在后台线程上执行占用大量 CPU 的工作。

